# ReproPilot: AI-Assisted Reproducibility Checker

This executable tutorial uses a deterministic HPC-aware rubric as the scoring source of truth. A local Ollama model receives actual repository evidence only to explain and prioritize findings.

In [1]:
from pathlib import Path
import json, tempfile, sys

# Make the repository root importable whether Jupyter starts from the repo root
# or from the notebooks/ directory.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'checker').is_dir():
        sys.path.insert(0, str(candidate.resolve()))
        break

from checker.reproducibility_checker import (
    assess_repository,
    build_evidence_package,
    discover_project_root,
    local_llm_recommendations,
    print_assessment,
)
print('Imports successful.')

Imports successful.


## 1. Assess this repository

In [2]:
PROJECT_PATH = discover_project_root()
DOMAIN = 'general'  # general, biomedical, climate, hpc-simulation
result = assess_repository(PROJECT_PATH, DOMAIN)
print_assessment(result)

Repository: /mnt/data/ReproPilot_Phase1/notebooks
Score: 0/100 (0.0%)
Interpretation: High reproducibility risk

[MISSING] Documentation                   0/15 Found: none
[MISSING] Dependency specification        0/15 Found: none
[MISSING] Reproducible environment        0/10 Found: none
[MISSING] HPC software stack              0/10 Found: none
[MISSING] Automated tests                 0/15 Found: none
[MISSING] Container recipe                0/15 Found: none
[MISSING] Experiment/provenance tracking  0/10 Found: none
[MISSING] License                         0/10 Found: none


## 2. Build evidence from actual files

In [3]:
evidence = build_evidence_package(PROJECT_PATH, result)
print('Files inventoried:', len(evidence['file_inventory']))
print('Snippets included:', list(evidence['selected_file_snippets']))
print(json.dumps(evidence['deterministic_score'], indent=2))

Files inventoried: 1
Snippets included: []
{
  "score": 0,
  "possible": 100,
  "percent": 0.0,
  "band": "High reproducibility risk"
}


## 3. Optional privacy-preserving local LLM explanation

Run `ollama pull gemma3:1b` and `ollama serve`. The model does not calculate or change the score.

In [4]:
llm_result = local_llm_recommendations(evidence)
print(json.dumps(llm_result, indent=2)[:5000])

{
  "ok": false,
  "message": "Local Ollama unavailable; static assessment completed: <urlopen error [Errno 111] Connection refused>"
}


## 4. Before/after demonstration

In [5]:
def write(root, rel, text='x'):
    p=root/rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text)
base=Path(tempfile.mkdtemp()); weak=base/'weak'; strong=base/'strong'; weak.mkdir(); strong.mkdir()
write(weak,'README.md','# Minimal example')
for rel in ['README.md','requirements.txt','environment.yml','spack.yaml','tests/test_smoke.py','apptainer.def','MLproject','LICENSE']:
    write(strong,rel)
print('BEFORE'); print_assessment(assess_repository(weak))
print('\nAFTER'); print_assessment(assess_repository(strong))

BEFORE
Repository: /tmp/tmp9yuxvlxc/weak
Score: 15/100 (15.0%)
Interpretation: High reproducibility risk

[PASS   ] Documentation                  15/15 Found: README.md
[MISSING] Dependency specification        0/15 Found: none
[MISSING] Reproducible environment        0/10 Found: none
[MISSING] HPC software stack              0/10 Found: none
[MISSING] Automated tests                 0/15 Found: none
[MISSING] Container recipe                0/15 Found: none
[MISSING] Experiment/provenance tracking  0/10 Found: none
[MISSING] License                         0/10 Found: none

AFTER
Repository: /tmp/tmp9yuxvlxc/strong
Score: 100/100 (100.0%)
Interpretation: Strong reproducibility

[PASS   ] Documentation                  15/15 Found: README.md
[PASS   ] Dependency specification       15/15 Found: requirements.txt
[PASS   ] Reproducible environment       10/10 Found: environment.yml
[PASS   ] HPC software stack             10/10 Found: spack.yaml
[PASS   ] Automated tests               

## Interpretation and limitations

File presence and non-empty content do not prove scientific correctness. Container builds, test quality, numerical stability, and non-deterministic HPC behavior require additional validation. LLM recommendations always require human review.